# DSCI 523 第一讲：readr / dplyr / tidyr

覆盖这一讲的全部 learning objectives。数据来自 `gapminder` 包（真实数据），
被 `make-data.R` 写成了各种格式，放在 `data/` 下。

**用法**：右上角 kernel 选 **R 4.6.1**，然后从头 `⇧Enter` 按到底。
每节末尾有「**动手**」小题，自己改代码试。

---

## 目录

1. 赋值符号 `<-`
2. `read_csv`：把 csv 读进来
3. `read_*` 家族和它的参数：应付不标准的文件
4. `write_csv`：把结果写出去
5. dplyr 六个动词：`select` `filter` `mutate` `arrange` `slice` `pull`
6. 管道 `|>`
7. 什么是 tidy data，好在哪、不好在哪
8. `pivot_longer` / `pivot_wider`：把不整洁的数据弄整洁

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


---
# 1. 赋值符号 `<-`

R 里把值绑到名字上用 **`<-`**，不是 `=`。

`=` 在赋值这件事上也能用，但 R 社区的约定是 `<-`，**523 的 lab 会按这个约定评分**。
真正的区别在于 `=` 在函数调用里另有含义（指定参数名），混用会产生歧义。

In [3]:
# 基本赋值
x <- 42
name <- "MDS-CL"
nums <- c(3, 1, 4, 1, 5)

x
name
nums

[1] 42

[1] "MDS-CL"

[1] 3 1 4 1 5

In [ ]:
# 为什么不用 = ：看这两行的区别
#
#   mean(x = c(1, 2, NA), na.rm = TRUE)   ← 这里的 = 是"指定参数"，不是赋值
#   mean(x <- c(1, 2, NA), na.rm = TRUE)  ← 这里的 <- 真的会创建一个叫 x 的变量
#
# 跑一下第二种，看看 x 被改掉了：
x <- 42
mean(x <- c(1, 2, NA), na.rm = TRUE)
x   # 已经不是 42 了

**快捷键**：VS Code 系的 R 扩展里，`Alt/⌥ + -` 会自动打出 ` <- `。

> **动手**：把上面 cell 的 `x <- 42` 改成 `x = 42`，再跑一次。
> 结果一样——说明在这个位置两者等价，区别只在函数调用内部。

---
# 2. `read_csv`：把 csv 读进来

`readr::read_csv()` 读逗号分隔文件。注意和 base R 的 `read.csv()`（点，不是下划线）区分：

| | `read_csv()`（readr） | `read.csv()`（base R） |
|---|---|---|
| 返回 | tibble | data.frame |
| 字符串 | 保持字符串 | 老版本会变 factor |
| 速度 | 快很多 | 慢 |
| 列类型 | 会打印它猜了什么 | 不说 |

**523 一律用 `read_csv()`。**

In [ ]:
gapminder <- read_csv("data/gapminder.csv")

gapminder

注意上面的输出分两部分：

1. **stderr 里的 `Rows: 60 Columns: 6` 和 Column specification** ——
   readr 在告诉你它把每一列猜成了什么类型。红色背景不是报错，是提示。
2. **tibble 本身** —— 每列名下面有 `<chr>` `<dbl>` 标明类型。

`<chr>` = character（字符串），`<dbl>` = double（小数），`<int>` = integer。

**养成习惯：读完必看类型。** 该是数字的列被读成 `<chr>`，说明文件里有脏数据。

In [ ]:
# 看结构的三个常用函数
glimpse(gapminder)

In [ ]:
# 只看前几行 / 后几行
head(gapminder, 3)
tail(gapminder, 3)

# 维度
dim(gapminder)
nrow(gapminder)
ncol(gapminder)

---
# 3. `read_*` 家族和参数：应付不标准的文件

现实里的文件很少是干净的 csv。readr 提供一族函数：

| 函数 | 用于 |
|---|---|
| `read_csv()` | 逗号分隔 |
| `read_tsv()` | 制表符分隔 |
| `read_delim()` | **任意分隔符**，用 `delim =` 指定 |
| `read_csv2()` | 分号分隔、逗号当小数点（欧洲格式） |
| `read_fwf()` | 固定宽度 |
| `read_lines()` | 一行一个字符串，不做任何解析 |

**选不出来的时候，先用 `read_lines()` 看看文件到底长什么样。**

In [ ]:
# 遇到陌生文件，第一步永远是这个：直接看原始行
read_lines("data/gapminder-messy.csv", n_max = 8)

看清楚了：前 4 行是注释，第 5 行开始才是数据，而且**没有表头**。
缺失值写成了 `N/A` 和 `-99`。

先看看不加任何参数会读成什么鬼样子：

In [ ]:
# 反面教材：直接读
bad <- read_csv("data/gapminder-messy.csv")
bad

一团糟：注释行被当成了表头，列名变成了 `# Gapminder extract`，
所有列都成了 `<chr>`。

用参数修正：

In [ ]:
messy <- read_csv(
  "data/gapminder-messy.csv",
  comment   = "#",                      # 以 # 开头的行整行跳过
  col_names = c("country", "continent", # 文件里没表头，自己给
                "year", "lifeExp", "pop", "gdpPercap"),
  na        = c("", "NA", "N/A", "-99") # 这些字符串都算缺失值
)

messy

现在 `lifeExp` 和 `gdpPercap` 是 `<dbl>`，缺失值是 `NA` 了。

几个最常用的参数：

| 参数 | 作用 |
|---|---|
| `comment = "#"` | 跳过以某字符开头的行 |
| `skip = 4` | 跳过开头 N 行（不管内容是什么） |
| `col_names = FALSE` | 文件没表头，列名自动叫 `X1` `X2`… |
| `col_names = c(...)` | 文件没表头，自己指定列名 |
| `na = c(...)` | 哪些字符串算缺失值 |
| `col_types = ...` | **强制**指定列类型，不让它猜 |
| `n_max = 100` | 只读前 N 行（试探大文件时用） |
| `delim = ";"` | `read_delim()` 专用，指定分隔符 |

In [ ]:
# skip 和 comment 的区别：skip 只认行数，comment 认内容
read_csv("data/gapminder-messy.csv", skip = 4,
         col_names = c("country","continent","year","lifeExp","pop","gdpPercap"),
         na = c("", "NA", "N/A", "-99"),
         n_max = 3)

In [ ]:
# 制表符分隔
read_tsv("data/gapminder.tsv", n_max = 3)

In [ ]:
# 任意分隔符：这个文件用分号
read_delim("data/gapminder-semicolon.txt", delim = ";", n_max = 3)

In [ ]:
# col_types：强制指定类型，别让它猜
#   c = character, d = double, i = integer, l = logical, D = date, _ = 跳过这一列
read_csv("data/gapminder.csv", col_types = "ccidid", n_max = 3)

> **动手**：把上面的 `col_types` 改成 `"cci_id"`，看 `lifeExp` 那一列会怎么样。
> （`_` 表示"读的时候直接扔掉这一列"。）

---
# 4. `write_csv`：把结果写出去

`readr::write_csv()`。和 base R 的 `write.csv()` 的关键区别：
**`write_csv` 默认不写行号**，`write.csv` 会多写一列没用的行号。

In [ ]:
# 先做一点处理，再写出去
canada <- filter(gapminder, country == "Canada")

write_csv(canada, "data/canada.csv")

# 确认写成功了：读回来看看
read_csv("data/canada.csv", n_max = 3)

In [ ]:
# 直接看写出去的原始文本，确认没有多余的行号列
read_lines("data/canada.csv", n_max = 3)

---
# 5. dplyr 六个动词

每个动词都是：**第一个参数是数据框，返回一个新的数据框**。
这个统一约定是管道 `|>` 能串起来的前提。

| 动词 | 干什么 | 改变的是 |
|---|---|---|
| `select()` | 挑**列** | 列数 |
| `filter()` | 挑**行**（按条件） | 行数 |
| `mutate()` | 新增/修改列 | 列数 |
| `arrange()` | 排序 | 行的顺序 |
| `slice()` | 挑**行**（按位置） | 行数 |
| `pull()` | 抽出一列变成**向量** | 类型（不再是数据框） |

## 5.1 `select()`：挑列

In [ ]:
# 要哪几列，就写哪几列
select(gapminder, country, year, lifeExp)

In [ ]:
# 减号 = 不要这一列
select(gapminder, -continent, -pop)

In [ ]:
# 冒号 = 连续一段列
select(gapminder, country:year)

# 辅助函数：按名字模式选
select(gapminder, starts_with("c"))
select(gapminder, ends_with("Exp"))
select(gapminder, contains("gdp"))

## 5.2 `filter()`：按条件挑行

In [ ]:
# 单条件
filter(gapminder, year == 2007)

In [ ]:
# 多条件：逗号分隔 = 且（AND）
filter(gapminder, year == 2007, lifeExp > 70)

# 显式的或（OR）用 |
filter(gapminder, country == "Canada" | country == "Japan")

### `%in%`：属于某个集合

要筛「这几个值之一」，别写一长串 `|`，用 `%in%`。

In [ ]:
# 啰嗦的写法
filter(gapminder, country == "Canada" | country == "Japan" | country == "China")

# %in% 的写法，等价但清爽
filter(gapminder, country %in% c("Canada", "Japan", "China"))

In [ ]:
# %in% 本身返回逻辑向量，跟 dplyr 无关，是 base R 的运算符
c("Canada", "Mars", "Japan") %in% c("Canada", "Japan", "China")

# 取反：前面加 !
filter(gapminder, !(country %in% c("Canada", "Japan"))) |> head(3)

⚠️ **常见坑**：`NA` 参与比较结果还是 `NA`，`filter()` 会把 `NA` 那些行**丢掉**。

In [ ]:
# messy 里 1952 年的 lifeExp 是 NA
filter(messy, lifeExp > 50)      # 1952 那 5 行不见了，一声不吭

# 要找出缺失值，必须用 is.na()
filter(messy, is.na(lifeExp))

## 5.3 `mutate()`：新增或修改列

In [ ]:
# 新增一列
mutate(gapminder, gdp_total = pop * gdpPercap)

In [ ]:
# 一次加多列，而且后面的可以用到前面刚建的
mutate(gapminder,
       gdp_total   = pop * gdpPercap,
       gdp_billion = gdp_total / 1e9,
       pop_million = round(pop / 1e6, 1))

In [ ]:
# 列名写成已有的名字 = 覆盖它
mutate(gapminder, lifeExp = round(lifeExp, 1))

## 5.4 `arrange()` 和 `desc()`：排序

In [ ]:
# 默认升序
arrange(gapminder, lifeExp)

In [ ]:
# desc() 包起来 = 降序
arrange(gapminder, desc(lifeExp))

In [ ]:
# 多个键：先按第一个排，第一个相同再按第二个
arrange(gapminder, country, desc(year))

## 5.5 `slice()`：按位置挑行

`filter()` 按**条件**挑，`slice()` 按**位置**挑。

In [ ]:
slice(gapminder, 1:5)        # 第 1 到 5 行
slice(gapminder, c(1, 10, 60)) # 指定几行
slice(gapminder, -(1:55))      # 除了前 55 行

In [ ]:
# slice 的一族变体，比自己 arrange 再 slice 更直观
slice_head(gapminder, n = 3)              # 前 3 行
slice_tail(gapminder, n = 3)              # 后 3 行
slice_max(gapminder, lifeExp, n = 3)      # lifeExp 最大的 3 行
slice_min(gapminder, lifeExp, n = 3)      # 最小的 3 行

## 5.6 `pull()`：抽出一列变成向量

前面所有动词返回的都是**数据框**。`pull()` 是唯一打破这点的——
它返回一个**向量**，所以它通常是管道链的最后一步。

In [ ]:
# 对比：select 返回单列数据框，pull 返回向量
select(gapminder, lifeExp) |> head(3)   # 还是 tibble
pull(gapminder, lifeExp)  |> head(3)    # 变成裸的数字向量

In [ ]:
# 为什么要它：后续要用的函数只吃向量，不吃数据框
mean(pull(gapminder, lifeExp))

# 常见用法：拿到一个去重的取值列表
gapminder |> pull(country) |> unique()

---
# 6. 管道 `|>`

`|>` 把左边的东西塞进右边函数的**第一个参数**。

```r
x |> f()        等价于   f(x)
x |> f(y)       等价于   f(x, y)
x |> f() |> g() 等价于   g(f(x))
```

为什么要它——看这三种写法解决同一个问题：
「加拿大和日本，2007 年，按预期寿命降序，只要国家和寿命两列」。

In [ ]:
# 写法 A：一层套一层。要从里往外读，反人类
select(
  arrange(
    filter(gapminder, country %in% c("Canada", "Japan"), year == 2007),
    desc(lifeExp)
  ),
  country, lifeExp
)

In [ ]:
# 写法 B：中间变量。能读，但要起一堆没意义的名字
tmp1 <- filter(gapminder, country %in% c("Canada", "Japan"), year == 2007)
tmp2 <- arrange(tmp1, desc(lifeExp))
tmp3 <- select(tmp2, country, lifeExp)
tmp3

In [ ]:
# 写法 C：管道。从上往下读，像念句子
gapminder |>
  filter(country %in% c("Canada", "Japan"), year == 2007) |>
  arrange(desc(lifeExp)) |>
  select(country, lifeExp)

**读法**：把 `|>` 念成「然后」。
「拿 gapminder，**然后**筛出这两国 2007 年，**然后**按寿命降序，**然后**只留两列。」

## `|>` vs `%>%`

会看到两种管道：

| | 来自 | 说明 |
|---|---|---|
| `|>` | **R 语言自带**（4.1+） | 不用装包。**523 用这个** |
| `%>%` | magrittr 包（tidyverse 会加载） | 老代码里全是它 |

日常用法几乎一样。差别在于 `%>%` 支持 `.` 占位符指代左边的值，`|>` 的占位符是 `_`
而且必须写成具名参数。写新代码就用 `|>`。

In [ ]:
# 两种管道，同样的结果
gapminder |> filter(year == 2007) |> nrow()
gapminder %>% filter(year == 2007) %>% nrow()

> **动手**：用管道写一句，求出 2007 年 5 个国家的**平均**预期寿命。
> 提示：`filter()` → `pull()` → `mean()`。

In [ ]:
# 你的答案写在这里

---
# 7. 什么是 tidy data

Hadley Wickham 的定义，三条规则：

1. **每一列是一个变量**（variable）
2. **每一行是一个观测**（observation）
3. **每个单元格是一个值**（value）

等价的说法：*一张表只装一种观测单位*。

看 `gapminder.csv`——它是整洁的：

In [ ]:
head(gapminder, 4)

对照三条规则：

- 列：`country` `continent` `year` `lifeExp` `pop` `gdpPercap`，**每个都是一个变量** ✓
- 行：一行 = 「某国某年」这一次观测 ✓
- 单元格：一格一个值 ✓

再看不整洁的版本：

In [ ]:
wide <- read_csv("data/life-expectancy-wide.csv")
wide

这张表**不整洁**。列名 `1952` `1957` `1962`… **本身是数据**（年份），不是变量名。

犯的是第 1 条：一个变量（year）被摊成了 12 列。

另一种不整洁：

In [ ]:
long <- read_csv("data/measurements-long.csv")
head(long, 8)

这张表也**不整洁**，但毛病相反：`measure` 列里塞了三个不同的变量
（lifeExp / pop / gdpPercap），`value` 列里是它们的值。

犯的是第 1 条的另一面：**多个变量被叠进了同一列**。
而且 `value` 这一列里，人口（上亿的整数）和寿命（几十的小数）混在一起，
类型上也说不通。

## 好处

| 好处 | 说明 |
|---|---|
| **函数接口统一** | 整个 tidyverse 都假定数据是整洁的。整洁了，`dplyr`/`ggplot2`/`tidyr` 全部即插即用 |
| **画图直接** | ggplot 的 `aes(x = , y = , colour = )` 每个都要映射到**一列**。不整洁就得先改造 |
| **分组统计直接** | `group_by(country)` 要求 country 是一列。摊成 12 列就没法分组 |
| **加变量不用改结构** | 多了一个指标，加一列就行，不用重排整张表 |
| **缺失值显式** | 该有的观测不存在，就是一个 `NA`，看得见 |

## 坏处

诚实地讲，整洁格式**不是永远更好**：

| 坏处 | 说明 |
|---|---|
| **占空间** | 长格式里 `country` 会重复几十上百遍。60 行的宽表变长表可能上千行 |
| **人看着累** | 给人读的表格（报告、论文附录）几乎都是宽的。「国家 × 年份」的矩阵一眼能扫，长表不行 |
| **录入不方便** | 手工填数据时，宽表更自然 |
| **有些方法要宽的** | 矩阵运算、相关矩阵、部分重复测量的统计模型、大部分机器学习的特征矩阵，要的都是宽格式 |
| **多余的 NA** | 长转宽时，本来不存在的组合会被显式填成 `NA` |

**结论**：整洁格式是**分析过程中的**标准形态，不是**存储和展示**的唯一正确形态。
所以才需要 `pivot_longer` / `pivot_wider` 在两者之间来回转。

---
# 8. `pivot_longer` / `pivot_wider`

| 函数 | 方向 | 什么时候用 |
|---|---|---|
| `pivot_longer()` | 宽 → 长 | **列名本身是数据**（比如年份当了列名） |
| `pivot_wider()` | 长 → 宽 | **一列里塞了多个变量** |

两个互为逆操作。

## 8.1 `pivot_longer()`：把摊开的列收回来

In [ ]:
wide   # 复习一下：12 个年份摊成了 12 列

In [ ]:
tidy_from_wide <- wide |>
  pivot_longer(
    cols      = -country,   # 除了 country，其他列全部收拢
    names_to  = "year",     # 原来的列名 → 放进这个新列
    values_to = "lifeExp"   # 原来的单元格值 → 放进这个新列
  )

tidy_from_wide

三个参数：

- `cols =` —— **哪些列要被收拢**。这里用 `-country` 表示「除了 country 之外全都要」。
  也可以写 `cols = "1952":"2007"` 或 `cols = !country`。
- `names_to =` —— 收拢后，**原列名**变成新列，叫什么名字。
- `values_to =` —— 收拢后，**原单元格值**变成新列，叫什么名字。

有个问题：`year` 现在是 `<chr>`（因为列名本来就是字符串）。用 `names_transform` 修：

In [ ]:
tidy_from_wide <- wide |>
  pivot_longer(
    cols = -country,
    names_to  = "year",
    values_to = "lifeExp",
    names_transform = list(year = as.integer)   # 转成整数
  )

tidy_from_wide

In [ ]:
# 验证：这下它是整洁的，可以直接画图了
tidy_from_wide |>
  ggplot(aes(x = year, y = lifeExp, colour = country)) +
  geom_line(linewidth = 1) +
  labs(x = "年份", y = "预期寿命", colour = "国家") +
  theme_minimal()

**这就是整洁的价值**：宽格式下这张图画不出来，因为 ggplot 的 x 轴要一列年份，
而宽表里年份是 12 个列名。

## 8.2 `pivot_wider()`：把叠起来的变量摊开

In [ ]:
head(long, 8)   # 复习：measure 列里塞了三个变量

In [ ]:
tidy_from_long <- long |>
  pivot_wider(
    names_from  = measure,   # 这一列的**取值**，变成新的列名
    values_from = value      # 这一列的值，去填新列的格子
  )

tidy_from_long

参数正好和 `pivot_longer` 对称：

- `names_from =` —— 哪一列的**取值**拿去当新列名
- `values_from =` —— 哪一列的值拿去填格子

现在三个指标各占一列，`lifeExp` 和 `pop` 也终于能有各自的类型了。

## 8.3 两者互逆

In [ ]:
# 转过去再转回来，应该回到原样
round_trip <- tidy_from_wide |>
  pivot_wider(names_from = year, values_from = lifeExp)

round_trip

# 和最初的 wide 比：列名的类型不同（一个是 "1952" 一个是 1952），
# 数值内容应该一致
all.equal(
  as.matrix(wide[, -1]),
  as.matrix(round_trip[, -1]),
  check.attributes = FALSE
)

## 8.4 `pivot_wider` 会造出 NA

长转宽时，原本**不存在**的组合会被显式填成 `NA`。这是长格式和宽格式的一个实质差异：
长格式里「没这条记录」，宽格式里就变成「有这个格子但值是 NA」。

In [ ]:
# 故意删掉一条记录，再转宽
long |>
  filter(!(country == "Brazil" & year == 2007 & measure == "pop")) |>
  pivot_wider(names_from = measure, values_from = value) |>
  filter(country == "Brazil")

---
# 9. 综合练习

把这一讲的东西串起来。每题先自己写，再看下面的参考答案。

**Q1** 从 `data/gapminder.csv` 读入，找出 2007 年预期寿命最高的 3 个国家，
只保留 `country` 和 `lifeExp` 两列。

**Q2** 算出每个国家 2007 年的 GDP 总量（`pop * gdpPercap`），单位换成"万亿"，
按降序排列。

**Q3** 从 `data/life-expectancy-wide.csv` 读入，转成整洁格式，
然后筛出 1952 年到 1972 年之间、预期寿命低于 50 的记录。

**Q4** 把 Q3 的结果写到 `data/low-life-exp.csv`。

In [ ]:
# Q1 你的答案

In [ ]:
# Q2 你的答案

In [ ]:
# Q3 你的答案

In [ ]:
# Q4 你的答案

---
## 参考答案

先自己做完再展开看。

In [ ]:
# Q1
read_csv("data/gapminder.csv", show_col_types = FALSE) |>
  filter(year == 2007) |>
  slice_max(lifeExp, n = 3) |>
  select(country, lifeExp)

In [ ]:
# Q2
gapminder |>
  filter(year == 2007) |>
  mutate(gdp_trillion = pop * gdpPercap / 1e12) |>
  arrange(desc(gdp_trillion)) |>
  select(country, gdp_trillion)

In [ ]:
# Q3
low <- read_csv("data/life-expectancy-wide.csv", show_col_types = FALSE) |>
  pivot_longer(cols = -country,
               names_to = "year", values_to = "lifeExp",
               names_transform = list(year = as.integer)) |>
  filter(year >= 1952, year <= 1972, lifeExp < 50) |>
  arrange(country, year)

low

In [ ]:
# Q4
write_csv(low, "data/low-life-exp.csv")

read_lines("data/low-life-exp.csv", n_max = 4)

---
# 速查表

```r
# --- 读 ---
read_csv(path)                         # 逗号
read_tsv(path)                         # 制表符
read_delim(path, delim = ";")          # 任意分隔符
read_lines(path, n_max = 10)           # 看原始行，判断格式用

  comment = "#"                        # 跳注释行
  skip = 4                             # 跳前 N 行
  col_names = FALSE / c("a","b")       # 没表头 / 自己给列名
  na = c("", "NA", "N/A", "-99")       # 哪些算缺失
  col_types = "ccidid"                 # c字符 d小数 i整数 l逻辑 D日期 _跳过
  n_max = 100                          # 只读前 N 行
  show_col_types = FALSE               # 别打印类型猜测

# --- 写 ---
write_csv(df, path)                    # 不写行号

# --- 挑列 ---
select(df, a, b)                       select(df, -a)
select(df, a:c)                        select(df, starts_with("c"))

# --- 挑行 ---
filter(df, x > 5, y == "a")            # 逗号 = 且
filter(df, x %in% c(1, 2, 3))          # 属于集合
filter(df, !(x %in% c(1, 2)))          # 不属于
filter(df, is.na(x))                   # 找缺失（NA 不能用 == 比）
slice(df, 1:5)                         slice_max(df, col, n = 3)
slice_head(df, n = 3)                  slice_min(df, col, n = 3)

# --- 改 ---
mutate(df, new = a * b)                # 新增/覆盖
arrange(df, x)                         arrange(df, desc(x))

# --- 取出向量 ---
pull(df, col)                          # 返回向量，不是数据框

# --- 管道 ---
x |> f() |> g()                        # 等价 g(f(x))

# --- 长宽转换 ---
pivot_longer(df, cols = -id,           # 宽→长：列名本身是数据
             names_to = "key", values_to = "val",
             names_transform = list(key = as.integer))

pivot_wider(df,                        # 长→宽：一列塞了多个变量
            names_from = key, values_from = val)
```

## tidy data 三条

1. 每列一个变量　2. 每行一个观测　3. 每格一个值

好处：tidyverse 全家即插即用、画图和分组直接、加变量不改结构。
坏处：占空间、人读着累、录入不方便、矩阵运算和 ML 特征矩阵要宽的、长转宽会造 NA。

**整洁是分析过程中的标准形态，不是存储和展示的唯一正确形态。**